In [3]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AQ') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'

gemini = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

API key looks good so far


In [5]:
links = fetch_website_links("https://vercel.com/")

In [ ]:
comparison_extraction_system_prompt = """
You are an expert business analyst.

You will be given the raw textual content of a company's website.

Your task is to extract ONLY the information that is useful for comparing this company with another company.

Focus on:
- Company Overview
- Products & Services
- Target Customers
- Industry
- Key Features
- Pricing (if available)
- Company Size (if available)
- Technology (if available)
- Unique Selling Points (USP)
- Customers/Partners (if available)
- Company Culture (if available)

Ignore:
- Navigation menus
- Footer content
- Copyright notices
- Privacy Policy
- Terms of Service
- Cookie notices
- Careers unless they reveal company culture
- Repeated information

Respond ONLY in JSON using this format:

{
  "company_overview": "",
  "products_services": "",
  "target_customers": "",
  "industry": "",
  "key_features": [],
  "pricing": "",
  "company_size": "",
  "technology": "",
  "unique_selling_points": [],
  "customers_partners": [],
  "company_culture": ""
}
"""

In [ ]:
def company_summar_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for comparison of one company to other, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": comparison_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    